# Two small applications I can trace by hand

Both experiments use two qubits. That keeps the state small enough to explain every output.

In [1]:
from math import pi

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

## Send two classical bits with one transmitted qubit

Superdense coding spends a Bell pair that Alice and Bob shared earlier. Alice encodes two bits on her half. X decides whether the two wires match, and Z decides the relative sign. Once Alice sends her qubit, Bob can read both choices by reversing the Bell preparation.

I use the left message bit for X and the right one for Z so Bob's printed q1q0 string matches the input. Try predicting the output for 10 before running all four.

In [2]:
def dense_code(message):
    circuit = QuantumCircuit(2)
    circuit.h(0)
    circuit.cx(0, 1)          # Alice holds q0, Bob holds q1.
    if message[0] == "1":
        circuit.x(0)
    if message[1] == "1":
        circuit.z(0)
    # Alice sends q0. Bob now holds both qubits and decodes.
    circuit.cx(0, 1)
    circuit.h(0)
    return circuit


for message in ("00", "01", "10", "11"):
    decoded = Statevector(dense_code(message))
    print(message, "->", dict(decoded.probabilities_dict(decimals=6)))
    assert round(decoded.probabilities()[int(message, 2)], 6) == 1

00 -> {np.str_('00'): np.float64(1.0)}
01 -> {np.str_('01'): np.float64(1.0)}
10 -> {np.str_('10'): np.float64(1.0)}
11 -> {np.str_('11'): np.float64(1.0)}


## Search four possibilities with one oracle call

Imagine a yes/no test that recognizes a winning two-bit string. In this tiny example I build the test myself so I can check every step. CZ marks 11 by flipping its phase. Right after that gate all four answers still have probability ¼.

Grover's diffusion step reflects each amplitude around their mean. The marked amplitudes here are [½, ½, ½, −½], with mean ¼. The reflection sends ½ to zero and −½ to one. The gate sequence below implements this reflection up to an overall sign.

In [3]:
search = QuantumCircuit(2)
search.h([0, 1])             # Four equal amplitudes.
search.cz(0, 1)              # Oracle marks |11> with a minus sign.
print("After marking:", Statevector(search).data.round(3))

search.h([0, 1])
search.x([0, 1])
search.cz(0, 1)
search.x([0, 1])
search.h([0, 1])             # Diffusion, up to a global phase.
answer = Statevector(search)
print("Final probabilities:", answer.probabilities().round(3))
assert round(answer.probabilities()[3], 6) == 1

After marking: [ 0.5+0.j  0.5+0.j  0.5+0.j -0.5+0.j]
Final probabilities: [0. 0. 0. 1.]


Try marking 10 instead. Wrap the **oracle** CZ between two X gates on q0 and change the check to index 2. Leave the diffusion step as written.

One iteration is enough here because exactly one of four answers is marked. For larger unstructured searches with one marked item, the number of oracle calls grows roughly with the square root of the number of candidates. Building the oracle and running the rest of the circuit also cost resources.